In [ ]:
! pip install deep-sort-realtime
! pip install ultralytics
! pip install datetime

In [ ]:
import datetime  # Library for handling date and time operations
from ultralytics import YOLO  # Library for loading and using the YOLO model
import cv2  # OpenCV library for image and video processing
from deep_sort_realtime.deepsort_tracker import DeepSort  # Library for the DeepSORT tracker
from google.colab.patches import cv2_imshow  # Library for displaying images in Google Colab

In [ ]:
def create_video_writer(video_cap, output_filename):
    # Function to create a video writer object for saving the output video
    frame_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(video_cap.get(cv2.CAP_PROP_FPS))
    fourcc = cv2.VideoWriter_fourcc(*'MP4V') # sets up the video writer with the MP4V codec
    writer = cv2.VideoWriter(output_filename, fourcc, fps, (frame_width, frame_height))
    return writer # This writer object is used later to save the processed video frames.

In [ ]:
CONFIDENCE_THRESHOLD = 0.8  # Confidence threshold for detecting objects
GREEN = (0, 255, 0)  # Color for drawing bounding boxes
WHITE = (255, 255, 255)  # Color for drawing text

video_cap = cv2.VideoCapture("/content/video.mp4")  # Initialize the video capture object to read the video
writer = create_video_writer(video_cap, "output.mp4")  # Initialize the video writer object to save the processed video

model = YOLO("yolov8n.pt")  # Load the pre-trained YOLOv8n model
tracker = DeepSort(max_age=50)  # Initialize the DeepSORT tracker

In [ ]:
while True:
    start = datetime.datetime.now()  # Record the start time

    ret, frame = video_cap.read()  # Read a frame from the video
    if not ret:
        break  # Exit the loop if no frame is read

    detections = model(frame)[0]  # Run the YOLO model on the frame to detect objects
    results = []

    for data in detections.boxes.data.tolist():
        confidence = data[4]  # Extract the confidence level of the detection
        if float(confidence) < CONFIDENCE_THRESHOLD:
            continue  # Ignore detections with low confidence

        # Get the bounding box coordinates and class ID
        xmin, ymin, xmax, ymax = int(data[0]), int(data[1]), int(data[2]), int(data[3])
        class_id = int(data[5])
        results.append([[xmin, ymin, xmax - xmin, ymax - ymin], confidence, class_id])

    tracks = tracker.update_tracks(results, frame=frame)
    for track in tracks:
        if not track.is_confirmed():
            continue  # Ignore unconfirmed tracks

        track_id = track.track_id  # Get the track ID
        ltrb = track.to_ltrb()  # Get the bounding box coordinates
        xmin, ymin, xmax, ymax = int(ltrb[0]), int(ltrb[1]), int(ltrb[2]), int(ltrb[3])
        # Draw the bounding box and the track ID on the frame
        cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), GREEN, 2)
        cv2.rectangle(frame, (xmin, ymin - 20), (xmin + 20, ymin), GREEN, -1)
        cv2.putText(frame, str(track_id), (xmin + 5, ymin - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.5, WHITE, 2)

    end = datetime.datetime.now()  # Record the end time
    print(f"Time to process 1 frame: {(end - start).total_seconds() * 1000:.0f} milliseconds")
    fps = f"FPS: {1 / (end - start).total_seconds():.2f}"  # Calculate and display the FPS
    cv2.putText(frame, fps, (50, 50), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 8)

    # Display the frame and write it to the output video
    cv2_imshow(frame)
    writer.write(frame)
    if cv2.waitKey(1) == ord("q"):
        break  # Exit the loop if 'q' key is pressed


video_cap.release()  # Release the video capture object
writer.release()  # Release the video writer object
cv2.destroyAllWindows()  # Close all OpenCV windows

## MOT20


from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import zipfile

zip_path = '/content/drive/MyDrive/MOT20.zip'  # path to your uploaded zip
extract_path = '/content/MOT20'                # where to extract

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("✅ Unzipped MOT20 to:", extract_path)


In [ ]:
import cv2
import os
import pandas as pd
import matplotlib.pyplot as plt

# Update this if your folder is named differently
dataset_path = '/content/MOT20/MOT20/train/MOT20-01'  # Adjust if yours is MOT20-02, etc.
img_folder = os.path.join(dataset_path, 'img1')
gt_path = os.path.join(dataset_path, 'gt', 'gt.txt')

# Load ground truth
gt = pd.read_csv(gt_path, header=None)
gt.columns = ['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'class', 'vis']

# Function to show frame with bounding boxes
def show_frame_with_boxes(frame_no):
    frame_file = os.path.join(img_folder, f"{str(frame_no).zfill(6)}.jpg")
    img = cv2.imread(frame_file)

    if img is None:
        print(f"Frame {frame_no} not found at {frame_file}")
        return

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    boxes = gt[gt['frame'] == frame_no]

    for _, row in boxes.iterrows():
        x, y, w, h, id_ = int(row.x), int(row.y), int(row.w), int(row.h), int(row.id)
        cv2.rectangle(img, (x, y), (x+w, y+h), (0, 255, 0), 2)
        cv2.putText(img, f'ID:{id_}', (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)

    plt.figure(figsize=(10,6))
    plt.imshow(img)
    plt.title(f"Frame {frame_no}")
    plt.axis('off')
    plt.show()

# Show a sample frame (e.g., frame 50)
show_frame_with_boxes(50)


## Algorithms


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!pip install ultralytics


In [ ]:
# Install required libs
# Only install what’s needed
!pip install deep-sort-realtime motmetrics opencv-python-headless

# YOLOv8 and ByteTrack already integrated via Ultralytics
!pip install -U ultralytics


# Clone tracking repos (optional, for reference/scripts)
!git clone https://github.com/abewley/sort.git
!git clone https://github.com/nwojke/deep_sort.git
!git clone https://github.com/ifzhang/ByteTrack.git


In [ ]:
# Fix for matplotlib backend error
import matplotlib
matplotlib.use('Agg')  # Use from ultralytics import YOLO
from deep_sort_realtime.deepsort_tracker import DeepSort
import cv2
import numpy as np
import motmetrics as mm
import matplotlib.pyplot as plt
from collections import defaultdict

# Load your custom-trained YOLOv8n model
model = YOLO("/content/drive/MyDrive/yolov8n_custom.pt")


# Initialize DeepSORT tracker
deep_sort = DeepSort()

# Video input and output setup
cap = cv2.VideoCapture('/content/video.mp4')
width, height = map(int, (cap.get(3), cap.get(4)))
fps = cap.get(cv2.CAP_PROP_FPS)
out = cv2.VideoWriter('tracked_output.mp4',
                      cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

# Logs for each tracker
logs = {
    "DeepSORT": defaultdict(list),
    "ByteTrack": defaultdict(list)
}

frame_num = 0

# Begin video loop
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_num += 1

    res = model(frame)[0]

    # Extract detections: [x1, y1, x2, y2, conf, class]
    dets = [[int(x1), int(y1), int(x2), int(y2), float(conf), int(cls)]
            for (x1, y1, x2, y2), conf, cls in zip(res.boxes.xyxy.tolist(),
                                                  res.boxes.conf.tolist(),
                                                  res.boxes.cls.tolist())]


    # DeepSORT
    ds_input = [[[x1, y1, x2 - x1, y2 - y1], conf, cls]
                for x1, y1, x2, y2, conf, cls in dets]
    for tr in deep_sort.update_tracks(ds_input, frame=frame):
        if not tr.is_confirmed():
            continue
        x1, y1, x2, y2 = map(int, tr.to_ltrb())
        logs["DeepSORT"][frame_num].append((tr.track_id, [x1, y1, x2, y2]))

    # ByteTrack
    bt_outputs = model.track(source=frame, tracker='bytetrack.yaml', persist=True)[0]
    if bt_outputs.boxes.id is not None:
        for i in range(len(bt_outputs.boxes)):
            x1, y1, x2, y2 = map(int, bt_outputs.boxes.xyxy[i].tolist())
            logs["ByteTrack"][frame_num].append(
                (int(bt_outputs.boxes.id[i]), [x1, y1, x2, y2]))

    out.write(frame)

cap.release()
out.release()





In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import motmetrics as mm
import sys
%matplotlib inline

# Fix for NumPy 2.0+ deprecated function used internally by motmetrics
if not hasattr(np, "asfarray"):
    np.asfarray = lambda a, dtype=None: np.asarray(a, dtype=np.float64 if dtype is None else dtype)

sys.path.append('/content/sort')  # Adjust path if needed

# ========== Evaluation Function ==========
def eval_metrics(track_log, name):
    acc = mm.MOTAccumulator(auto_id=True)

    for frame, objs in sorted(track_log.items()):
        g_ids = [tid for tid, _ in objs]
        g_boxes = [box for _, box in objs]
        tr_ids = g_ids
        tr_boxes = g_boxes

        g_boxes = np.asarray(g_boxes, dtype=np.float32)
        tr_boxes = np.asarray(tr_boxes, dtype=np.float32)

        dist = mm.distances.iou_matrix(g_boxes, tr_boxes, max_iou=0.5)
        acc.update(g_ids, tr_ids, dist)

    mh = mm.metrics.create()
    metrics = ['mota', 'idf1', 'num_switches', 'mostly_tracked', 'mostly_lost']
    summary = mh.compute(acc, metrics=metrics, name=name)

    print(f"\n===== {name} Evaluation Metrics =====")
    print(summary.to_string(index=False))



# ========== Run Evaluation ==========
for tracker_name in logs:
    eval_metrics(logs[tracker_name], tracker_name)



| Tracker       | Strengths                                      | Weaknesses                         |
| ------------- | ---------------------------------------------- | ---------------------------------- |
| **SORT**      | Fast (\~200 FPS), simple to set up             | Cannot re-ID; high ID switching    |
| **DeepSORT**  | Robust to occlusions, retains identity         | Slower; requires appearance model  |
| **ByteTrack** | High recall by using low-confidence detections | More complex; no visual embeddings |


In [ ]:
import cv2
import os

def visualize_tracking(video_path, track_log, output_path, tracker_name):
    cap = cv2.VideoCapture(video_path)
    frame_idx = 0

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Output video writer
    out = cv2.VideoWriter(
        os.path.join(output_path, f"{tracker_name}_tracked.mp4"),
        cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height)
    )

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx in track_log:
            for tid, box in track_log[frame_idx]:
                x1, y1, x2, y2 = map(int, box)
                color = (0, 255, 0)  # Green box
                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                cv2.putText(frame, f'ID {tid}', (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        cv2.putText(frame, f'{tracker_name} - Frame {frame_idx}', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 0, 0), 2)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    print(f"{tracker_name} tracking video saved to {output_path}")


In [ ]:
video_path = "/content/video.mp4"  # Path to your original input video
output_dir = "output_videos"    # Ensure this directory exists

os.makedirs(output_dir, exist_ok=True)

visualize_tracking(video_path, logs["DeepSORT"], output_dir, "DeepSORT")
visualize_tracking(video_path, logs["ByteTrack"], output_dir, "ByteTrack")
